# Customer Churn Analysis and Retention Strategy
**Author:** Sanman Kadam  
**Status:** Complete  
**License:** MIT  

---

## 1. Business Problem
A telecom company operating across major regional circles in India is experiencing customer attrition (churn), leading to substantial revenue loss and increased subscriber acquisition costs. In the highly competitive Indian telecom market, retaining high-value subscribers is critical for maintaining market share and profitability.

The objective of this project is to analyze historical subscriber data, identify key drivers of customer churn, segment regional circles based on risk and revenue value, and build machine learning models to predict subscriber decline at a circle level. These insights will drive action-oriented retention strategies with quantifiable business impact.

## 2. Project Objectives
- **Descriptive Analysis:** Understand subscriber trends and regional differences across circles.
- **Revenue Risk Mapping:** Quantify the financial impact of subscriber attrition across different regions and connection types.
- **Risk Segmentation:** Categorize circles into risk tiers (Low, Medium, High, Critical) using a composite risk score.
- **Predictive Modeling:** Train and evaluate classification models (Logistic Regression, Random Forest, Decision Trees, etc.) to predict subscriber decline.
- **Strategic Recommendations:** Propose specific, ROI-backed actions to minimize churn and protect revenue.

---


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
import os
paths = ["Telecom_Features.csv", "notebooks/Telecom_Features.csv", "../data/Telecom_Features.csv", "data/Telecom_Features.csv"]
for path in paths:
    if os.path.exists(path):
        df = pd.read_csv(path)
        break


In [3]:
df.head()

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30196 entries, 0 to 30195
Data columns (total 29 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   year                           30196 non-null  int64  
 1   month                          30196 non-null  object 
 2   circle                         30196 non-null  object 
 3   type_of_connection             30196 non-null  object 
 4   service_provider               30196 non-null  object 
 5   value                          30196 non-null  float64
 6   unit                           30196 non-null  object 
 7   notes                          51 non-null     object 
 8   month_num                      30196 non-null  int64  
 9   date                           30196 non-null  object 
 10  subscribers_lag_1              30196 non-null  float64
 11  subscribers_lag_3              30196 non-null  float64
 12  subscribers_lag_6              30196 non-null 

In [5]:
df.describe()

In [6]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Check top rows and data info
print(df.head())
print(df.info())

   year     month  ... market_size_category operator_geographic_diversity
0  2009   January  ...             399688.0                          27.0
1  2009  February  ...             399688.0                          27.0
2  2009     March  ...             399688.0                          27.0
3  2009     April  ...             399688.0                          27.0
4  2009       May  ...             399688.0                          27.0

[5 rows x 29 columns]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30196 entries, 0 to 30195
Data columns (total 29 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   year                           30196 non-null  int64  
 1   month                          30196 non-null  object 
 2   circle                         30196 non-null  object 
 3   type_of_connection             30196 non-null  object 
 4   service_provider               30196 non-null  object 
 5  

In [7]:
# Numeric summary
numeric_cols = df.select_dtypes(include=np.number).columns
print(df[numeric_cols].describe().T)

# Count of missing values
print(df.isna().sum())

# Unique values in key categorical columns
print(df['service_provider'].nunique(), df['circle'].nunique())


                                 count  ...           max
year                           30196.0  ...  2.025000e+03
value                          30196.0  ...  2.955508e+08
month_num                      30196.0  ...  1.200000e+01
subscribers_lag_1              30196.0  ...  2.955508e+08
subscribers_lag_3              30196.0  ...  2.955508e+08
subscribers_lag_6              30196.0  ...  2.955508e+08
subscribers_lag_12             30196.0  ...  2.955508e+08
mom_growth                     30196.0  ...  1.026224e+04
yoy_growth                     30196.0  ...  1.287314e+04
growth_volatility_3            30196.0  ...  7.256502e+03
growth_volatility_6            30196.0  ...  7.256502e+03
growth_volatility_12           30196.0  ...  7.256502e+03
trend_12m                      30196.0  ...  2.699598e+08
total_circle_subscribers       30196.0  ...  3.637702e+08
market_share                   30196.0  ...  1.000000e+00
market_rank                    30196.0  ...  1.600000e+01
share_gap_lead

In [8]:
# Fill missing growth values with median (more realistic)
df['mom_growth'] = df['mom_growth'].fillna(df['mom_growth'].median())
df['yoy_growth'] = df['yoy_growth'].fillna(df['yoy_growth'].median())

In [9]:
# Check missing values
print("Missing values after cleaning:\n", df.isna().sum())

Missing values after cleaning:
 year                                 0
month                                0
circle                               0
type_of_connection                   0
service_provider                     0
value                                0
unit                                 0
notes                            30145
month_num                            0
date                                 0
subscribers_lag_1                    0
subscribers_lag_3                    0
subscribers_lag_6                    0
subscribers_lag_12                   0
mom_growth                           0
yoy_growth                           0
growth_volatility_3                  0
growth_volatility_6                  0
growth_volatility_12                 0
trend_12m                            0
total_circle_subscribers             0
market_share                         0
market_rank                          0
share_gap_leader                     0
relative_performance            

In [10]:
# -----------------------
# Numeric Summary
# -----------------------
numeric_cols = df.select_dtypes(include=np.number).columns
print("\nNumeric Summary:\n", df[numeric_cols].describe().T)


Numeric Summary:
                                  count  ...           max
year                           30196.0  ...  2.025000e+03
value                          30196.0  ...  2.955508e+08
month_num                      30196.0  ...  1.200000e+01
subscribers_lag_1              30196.0  ...  2.955508e+08
subscribers_lag_3              30196.0  ...  2.955508e+08
subscribers_lag_6              30196.0  ...  2.955508e+08
subscribers_lag_12             30196.0  ...  2.955508e+08
mom_growth                     30196.0  ...  1.026224e+04
yoy_growth                     30196.0  ...  1.287314e+04
growth_volatility_3            30196.0  ...  7.256502e+03
growth_volatility_6            30196.0  ...  7.256502e+03
growth_volatility_12           30196.0  ...  7.256502e+03
trend_12m                      30196.0  ...  2.699598e+08
total_circle_subscribers       30196.0  ...  3.637702e+08
market_share                   30196.0  ...  1.000000e+00
market_rank                    30196.0  ...  1.600000

In [11]:
# -----------------------
#  Univariate Analysis
# -----------------------
for col in numeric_cols:
    plt.figure(figsize=(12,4))

    # Histogram
    plt.subplot(1,2,1)
    sns.histplot(df[col], bins=50, kde=True)
    plt.title(f'Distribution of {col}')

    # Boxplot
    plt.subplot(1,2,2)
    sns.boxplot(y=df[col])
    plt.title(f'Boxplot of {col}')

    plt.tight_layout()
    plt.show()


In [12]:
# Set style
sns.set(style="whitegrid")

# Columns to plot
cols_to_plot = ['value', 'market_share', 'trend_12m']

# Create subplots
plt.figure(figsize=(18, 6))

for i, col in enumerate(cols_to_plot, 1):
    plt.subplot(1, 3, i)
    sns.boxplot(y=df[col])
    plt.title(f'Boxplot of {col}')
    plt.ylabel(col)

plt.tight_layout()
plt.show()

In [13]:
# For heavily skewed columns like 'value', use log scale
plt.figure(figsize=(6,3))
sns.histplot(np.log1p(df['value']), bins=50, kde=True)
plt.title('Distribution of log(Value)')
plt.show()

In [14]:
# -----------------------
# Categorical Columns Analysis
# -----------------------
categorical_cols = ['circle_type', 'is_wireless', 'service_provider']

for col in categorical_cols:
    plt.figure(figsize=(8,4))
    sns.countplot(y=col, data=df, order=df[col].value_counts().index)
    plt.title(f'Counts of {col}')
    plt.show()

In [15]:
# Unique counts
print("\nUnique values:")
for col in categorical_cols:
    print(f"{col}: {df[col].nunique()}")


Unique values:
circle_type: 2
is_wireless: 1
service_provider: 33


In [16]:

# -----------------------------
# TIME-BASED ANALYSIS (TOP PROVIDERS)
# -----------------------------

In [17]:
# Top 10 providers by total subscribers

# Step 1: Create proper datetime column for time-series order
df['year_month'] = pd.to_datetime(df['year'].astype(str) + '-' + df['month_num'].astype(str) + '-01')

# Step 2: Find top 5 providers
top_10_providers = df.groupby('service_provider')['value'].sum().sort_values(ascending=False).head(10).index.tolist()
print("Top 10 providers:", top_10_providers)

# Step 3: Filter to only those top 5 providers
df_top10 = df[df['service_provider'].isin(top_10_providers)]

# Step 4: Aggregate total subscribers per provider per month
df_agg = df_top10.groupby(['year_month', 'service_provider'])['value'].sum().reset_index()

# Step 5: Plot
plt.figure(figsize=(12,6))
sns.lineplot(data=df_agg, x='year_month', y='value', hue='service_provider')
plt.title('Monthly Subscribers for Top 10 Providers')
plt.xlabel('Year-Month')
plt.ylabel('Subscribers')
plt.xticks(rotation=45)
plt.show()


Top 10 providers: ['Bharti Airtel', 'Reliance Jio', 'Vodafone Idea', 'BSNL', 'Vodafone', 'Idea', 'Reliance Communications', 'Bharti Airtel (Including Tata Tele.)', 'Tata Teleservices', 'Aircel']


In [18]:
# Top 15 providers by total subscribers

# Step 1: Create proper datetime column for time-series order
df['year_month'] = pd.to_datetime(df['year'].astype(str) + '-' + df['month_num'].astype(str) + '-01')

# Step 2: Find top 5 providers
top_5_providers = df.groupby('service_provider')['value'].sum().sort_values(ascending=False).head(5).index.tolist()
print("Top 5 providers:", top_5_providers)

# Step 3: Filter to only those top 5 providers
df_top5 = df[df['service_provider'].isin(top_5_providers)]

# Step 4: Aggregate total subscribers per provider per month
df_agg = df_top5.groupby(['year_month', 'service_provider'])['value'].sum().reset_index()

# Step 5: Plot
plt.figure(figsize=(12,6))
sns.lineplot(data=df_agg, x='year_month', y='value', hue='service_provider')
plt.title('Monthly Subscribers for Top 5 Providers')
plt.xlabel('Year-Month')
plt.ylabel('Subscribers')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


Top 5 providers: ['Bharti Airtel', 'Reliance Jio', 'Vodafone Idea', 'BSNL', 'Vodafone']


In [19]:
# Market share distribution

plt.figure(figsize=(6,4))
sns.histplot(df['market_share'], bins=30, kde=True)
plt.title('Market Share Distribution')
plt.show()

In [20]:

# -----------------------------
#  CORRELATION ANALYSIS
# -----------------------------

# Correlation heatmap
key_feats = ['value', 'mom_growth', 'yoy_growth', 'trend_12m',
             'market_share', 'share_gap_leader', 'relative_performance']

plt.figure(figsize=(8,6))
sns.heatmap(df[key_feats].corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

In [21]:
# Print top correlated pairs
corr_matrix = df[key_feats].corr().abs().unstack().sort_values(ascending=False)
corr_pairs = corr_matrix[(corr_matrix < 1)].head(5)
print("\nTop correlated feature pairs:\n", corr_pairs)


Top correlated feature pairs:
 relative_performance  share_gap_leader        0.860739
share_gap_leader      relative_performance    0.860739
relative_performance  market_share            0.849005
market_share          relative_performance    0.849005
share_gap_leader      market_share            0.787434
dtype: float64


In [22]:
# -----------------------------
# BIVARIATE ANALYSIS
# -----------------------------


# Provider vs Circle Type
plt.figure(figsize=(10,6))
sns.barplot(data=df, x='service_provider', y='value', hue='circle_type',
            estimator='sum', ci=None)
plt.title('Subscribers by Provider and Circle Type')
plt.xticks(rotation=45)
plt.show()


<string>:8: FutureWarning: 

The `ci` parameter is deprecated. Use `errorbar=None` for the same effect.



In [23]:
# -----------------------------
#  CHURN TREND ANALYSIS
# -----------------------------


# Identify negative growth circles
neg_growth = df[df['mom_growth'] < 0].groupby('circle')['mom_growth'].mean().sort_values()
print("\nCircles with negative average MoM growth:\n", neg_growth.head(10))


Circles with negative average MoM growth:
 circle
Tamil Nadu         -0.125708
Haryana            -0.064901
Himachal Pradesh   -0.062972
Kerala             -0.056527
Gujarat            -0.051443
Madhya Pradesh     -0.048626
West Bengal        -0.046673
Delhi              -0.045839
Bihar              -0.045086
Mumbai             -0.044782
Name: mom_growth, dtype: float64


In [24]:
# Top 5 churn circles
top_churn_circles = neg_growth.head(5).index.tolist()

# Filter for churn trend visualization
df_churn = df[df['circle'].isin(top_churn_circles)]

plt.figure(figsize=(12,6))
sns.lineplot(data=df_churn, x='year_month', y='mom_growth', hue='circle')
plt.title('Churn Trend (Negative MoM Growth Circles)')
plt.xlabel('Year-Month')
plt.ylabel('MoM Growth')
plt.xticks(rotation=45)
plt.show()

In [25]:
# Compute average MoM growth per circle
mom_circle = df.groupby('circle')['mom_growth'].mean().sort_values()

# Color: red if negative, green if positive
colors = ['red' if x < 0 else 'green' for x in mom_circle]

# Plot
plt.figure(figsize=(12,6))
sns.barplot(x=mom_circle.index, y=mom_circle.values, palette=colors)
plt.xticks(rotation=90)
plt.ylabel('Average MoM Growth')
plt.title('Average Month-on-Month Growth per Circle')
plt.tight_layout()
plt.show()

<string>:9: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.



In [26]:
# Top 10 circles with negative average MoM growth
top_negative = mom_circle.nsmallest(10)

plt.figure(figsize=(8,5))
sns.barplot(x=top_negative.values, y=top_negative.index, palette='Reds_r')
plt.xlabel('Average MoM Growth')
plt.ylabel('Circle')
plt.title('Top 10 Circles with Negative Month-on-Month Growth')
plt.tight_layout()
plt.show()


<string>:5: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.



In [27]:
# Key Insights
# -----------------------
# Top providers by subscribers
top_ops = df.groupby('service_provider')['value'].sum().sort_values(ascending=False).head(10)
print("\nTop 10 Providers by Total Subscribers:\n", top_ops)

# Metro vs Non-Metro analysis
metro_summary = df.groupby('circle_type')['value'].sum()
print("\nTotal subscribers Metro vs Non-Metro:\n", metro_summary)

# Negative growth circles
neg_growth = df[df['mom_growth'] < 0].groupby('circle')['mom_growth'].mean().sort_values()
print("\nCircles with negative average MoM growth:\n", neg_growth.head(10))


Top 10 Providers by Total Subscribers:
 service_provider
Bharti Airtel                           4.448258e+10
Reliance Jio                            3.626207e+10
Vodafone Idea                           2.201273e+10
BSNL                                    1.880140e+10
Vodafone                                1.775930e+10
Idea                                    1.529843e+10
Reliance Communications                 1.181269e+10
Bharti Airtel (Including Tata Tele.)    7.699103e+09
Tata Teleservices                       7.310867e+09
Aircel                                  5.307474e+09
Name: value, dtype: float64

Total subscribers Metro vs Non-Metro:
 circle_type
Metro        2.026674e+10
Non-Metro    1.758159e+11
Name: value, dtype: float64

Circles with negative average MoM growth:
 circle
Tamil Nadu         -0.125708
Haryana            -0.064901
Himachal Pradesh   -0.062972
Kerala             -0.056527
Gujarat            -0.051443
Madhya Pradesh     -0.048626
West Bengal        -0.04667

In [28]:
import matplotlib.pyplot as plt
import seaborn as sns

# Make sure top 5 providers list exists
top_5_providers = df.groupby('service_provider')['value'].sum().sort_values(ascending=False).head(5).index.tolist()

# -------------------------------
# 1️⃣ Boxplot per Circle
# -------------------------------
plt.figure(figsize=(15,6))
sns.boxplot(data=df, x='circle', y='value')
plt.xticks(rotation=90)
plt.title('Distribution of Subscribers per Circle')
plt.xlabel('Circle')
plt.ylabel('Subscribers')
plt.tight_layout()
plt.savefig('boxplot_circle.png')  # Save figure for slides
plt.show()

# -------------------------------
# 2️⃣ Boxplot per Top Providers
# -------------------------------
plt.figure(figsize=(12,6))
sns.boxplot(data=df[df['service_provider'].isin(top_5_providers)],
            x='service_provider', y='value')
plt.xticks(rotation=45)
plt.title('Distribution of Subscribers per Top 5 Providers')
plt.xlabel('Service Provider')
plt.ylabel('Subscribers')
plt.tight_layout()
plt.savefig('boxplot_top5_providers.png')
plt.show()

# -------------------------------
# 3️⃣ Violin Plot per Top Providers
# -------------------------------
plt.figure(figsize=(12,6))
sns.violinplot(data=df[df['service_provider'].isin(top_5_providers)],
               x='service_provider', y='value')
plt.xticks(rotation=45)
plt.title('Subscribers Distribution per Top 5 Providers (Violin Plot)')
plt.xlabel('Service Provider')
plt.ylabel('Subscribers')
plt.tight_layout()
plt.savefig('violin_top5_providers.png')
plt.show()


In [29]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
try:
    from IPython.display import display
except ImportError:
    display = print


In [30]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
import matplotlib.pyplot as plt
import seaborn as sns

In [31]:
import os
paths = ["Cleaned_Telecom_Subscriptions.csv", "notebooks/Cleaned_Telecom_Subscriptions.csv", "../data/Cleaned_Telecom_Subscriptions.csv", "data/Cleaned_Telecom_Subscriptions.csv"]
for path in paths:
    if os.path.exists(path):
        data = pd.read_csv(path)
        break


<string>:5: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.


In [32]:
print("Dataset shape:", data.shape)
display(data.head())

Dataset shape: (70728, 8)
   year  month          circle  ...     value                      unit notes
0  2025  April  Andhra Pradesh  ...  33965795  value in absolute number   NaN
1  2025  April           Assam  ...  12314102  value in absolute number   NaN
2  2025  April           Bihar  ...  40967773  value in absolute number   NaN
3  2025  April           Delhi  ...  18877637  value in absolute number   NaN
4  2025  April         Gujarat  ...  12401101  value in absolute number   NaN

[5 rows x 8 columns]


In [33]:
cat_cols = data.select_dtypes(include=['object']).columns

le = LabelEncoder()
for col in cat_cols:
    data[col] = le.fit_transform(data[col].astype(str))

print("\nMissing values per column:")
print(data.isnull().sum())

data.fillna(data.median(), inplace=True)


Missing values per column:
year                  0
month                 0
circle                0
type_of_connection    0
service_provider      0
value                 0
unit                  0
notes                 0
dtype: int64


In [34]:
target_col = 'circle'  
if target_col not in data.columns:
    raise ValueError(f"The column '{target_col}' is not found. Please update 'target_col' to your target variable name.")

X = data.drop(columns=[target_col])
y = data[target_col]

In [35]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [36]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [37]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

/opt/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/opt/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/opt/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/opt/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:336: RuntimeWarning: divide by zero encountered in matmul
  grad[:, :n_features] = grad_pointwise.T @ X + l2_reg_strength * weights
/opt/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:336: RuntimeWarning: overflow encountered in matmul
  grad[:

In [38]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

/opt/miniconda3/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/opt/miniconda3/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/opt/miniconda3/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/opt/miniconda3/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/opt/miniconda3/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/opt/miniconda3/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


In [39]:
plt.figure(figsize=(5,4))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

<string>:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


In [40]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

n_classes = len(np.unique(y_test))

plt.figure(figsize=(7, 6))

if n_classes == 2:
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc_val = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"ROC Curve (AUC = {roc_auc_val:.2f})")

else:
    y_test_bin = label_binarize(y_test, classes=np.unique(y_test))
    
    if y_proba.ndim == 1:  # If model returned 1D probabilities
        y_proba_bin = np.tile(y_proba.reshape(-1, 1), (1, n_classes))
    else:
        y_proba_bin = y_proba

    for i in range(n_classes):
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_proba_bin[:, i])
        roc_auc_i = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f"Class {i} (AUC = {roc_auc_i:.2f})")

plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend(loc="lower right")
plt.show()

<string>:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


In [41]:
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Classification Report:
              precision    recall  f1-score   support

           0       0.06      0.54      0.11      1020
           1       0.00      0.00      0.00         3
           2       0.00      0.00      0.00         3
           3       0.00      0.00      0.00        18
           4       0.06      0.02      0.03       885
           5       0.00      0.00      0.00        17
           6       0.01      0.00      0.00       824
           7       0.04      0.17      0.07       863
           9       0.06      0.14      0.09        57
          10       0.00      0.00      0.00        13
          11       0.05      0.01      0.02       940
          12       0.00      0.00      0.00       927
          13       0.05      0.10      0.07       872
          14       0.00      0.00      0.00       931
          15       0.06      0.02      0.03       838
          16       0.00      0.00      0.00        17
          17       0.02      0.00      0.00       929
   

/opt/miniconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/miniconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/miniconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [42]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=5),
        "KNN": KNeighborsClassifier()
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results[name] = acc
    print(f"\n{name} Accuracy: {acc:.4f}")
    print(classification_report(y_test, y_pred))



Logistic Regression Accuracy: 0.0525
              precision    recall  f1-score   support

           0       0.06      0.54      0.11      1020
           1       0.00      0.00      0.00         3
           2       0.00      0.00      0.00         3
           3       0.00      0.00      0.00        18
           4       0.06      0.02      0.03       885
           5       0.00      0.00      0.00        17
           6       0.01      0.00      0.00       824
           7       0.04      0.17      0.07       863
           9       0.06      0.14      0.09        57
          10       0.00      0.00      0.00        13
          11       0.05      0.01      0.02       940
          12       0.00      0.00      0.00       927
          13       0.05      0.10      0.07       872
          14       0.00      0.00      0.00       931
          15       0.06      0.02      0.03       838
          16       0.00      0.00      0.00        17
          17       0.02      0.00      0.00

/opt/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/opt/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/opt/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/opt/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:336: RuntimeWarning: divide by zero encountered in matmul
  grad[:, :n_features] = grad_pointwise.T @ X + l2_reg_strength * weights
/opt/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:336: RuntimeWarning: overflow encountered in matmul
  grad[:

In [43]:
result_df = pd.DataFrame(list(results.items()), columns=['Model', 'Accuracy']).sort_values(by='Accuracy', ascending=False)
print("\nModel Comparison:\n", result_df)


Model Comparison:
                  Model  Accuracy
1        Decision Tree  0.337433
2        Random Forest  0.196522
3    Gradient Boosting  0.090532
4                  KNN  0.076959
0  Logistic Regression  0.052500


In [44]:
plt.figure(figsize=(10,5))
sns.barplot(x='Accuracy', y='Model', data=result_df, palette='viridis')
plt.title("Model Accuracy Comparison")
plt.show()

<string>:2: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

<string>:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


In [45]:
best_model_name = result_df.iloc[0]['Model']
best_model = models[best_model_name]
print(f"\n Best Model: {best_model_name} with Accuracy = {result_df.iloc[0]['Accuracy']:.4f}")


 Best Model: Decision Tree with Accuracy = 0.3374


In [46]:
import joblib
joblib.dump(best_model, f"best_model_{best_model_name.replace(' ', '_')}.pkl")
print(f"Model saved as best_model_{best_model_name.replace(' ', '_')}.pkl")

Model saved as best_model_Decision_Tree.pkl


In [47]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=5),
        "KNN": KNeighborsClassifier()
}

evaluation = []

for name, model in models.items():
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    
    auc = np.nan
    if hasattr(model, "predict_proba"):
        try:
            y_proba = model.predict_proba(X_test)
            if len(np.unique(y_test)) == 2:
                auc = roc_auc_score(y_test, y_proba[:, 1])
            elif len(np.unique(y_test)) > 2:
                auc = roc_auc_score(y_test, y_proba, multi_class='ovr')
        except Exception:
            auc = np.nan 

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

    evaluation.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1,
        'AUC': auc
    })


/opt/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/opt/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/opt/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/opt/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:336: RuntimeWarning: divide by zero encountered in matmul
  grad[:, :n_features] = grad_pointwise.T @ X + l2_reg_strength * weights
/opt/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:336: RuntimeWarning: overflow encountered in matmul
  grad[:

In [48]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import numpy as np
import pandas as pd

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=5),
        "KNN": KNeighborsClassifier()
}

evaluation = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    auc = np.nan
    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)
        try:
            if len(np.unique(y_test)) == 2:
                auc = roc_auc_score(y_test, y_proba[:, 1])
            else:
                if y_proba.shape[1] == len(np.unique(y_test)):
                    auc = roc_auc_score(y_test, y_proba, multi_class='ovr')
        except ValueError:
            auc = np.nan

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

    evaluation.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1,
        'AUC': auc
    })
    
eval_df = pd.DataFrame(evaluation).sort_values(by='Accuracy', ascending=False)
display(eval_df)


                 Model  Accuracy  Precision    Recall  F1-Score  AUC
1        Decision Tree  0.336114   0.392971  0.336114  0.354446  NaN
2        Random Forest  0.200339   0.199956  0.200339  0.200004  NaN
3    Gradient Boosting  0.090532   0.133147  0.090532  0.080966  NaN
4                  KNN  0.076959   0.094721  0.076959  0.072660  NaN
0  Logistic Regression  0.052500   0.034484  0.052500  0.025344  NaN


/opt/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/opt/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/opt/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/opt/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:336: RuntimeWarning: divide by zero encountered in matmul
  grad[:, :n_features] = grad_pointwise.T @ X + l2_reg_strength * weights
/opt/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:336: RuntimeWarning: overflow encountered in matmul
  grad[:

In [49]:
best_model_name = eval_df.iloc[0]['Model']
print("best Model Selected:", best_model_name)

best Model Selected: Decision Tree
